# 03 — Relaxed R1CS: Why Naive Folding Fails

This notebook demonstrates the **core insight** of Nova: why you can't naively combine two R1CS instances, and how **Relaxed R1CS** fixes it.

## The Goal

Given two R1CS instances $(\mathbf{x}_1, \mathbf{w}_1)$ and $(\mathbf{x}_2, \mathbf{w}_2)$ for the same circuit, we want to combine them into **one** instance that's cheaper to prove.

## The Natural Attempt: Random Linear Combination

Pick random $r$ and set $\mathbf{z}' = \mathbf{z}_1 + r \cdot \mathbf{z}_2$. Does $\mathbf{z}'$ satisfy R1CS?

**Nova paper reference:** Section 4, motivation for Construction 4

In [ ]:
import numpy as np
from nano_nova.field import GF, random_field_element
from nano_nova.examples import fibonacci_step_circuit, fibonacci_step_fn, fibonacci_witness_fn

shape = fibonacci_step_circuit()

# Two valid Fibonacci steps
z1_in = GF(np.array([1, 1]))
z1_out = fibonacci_step_fn(z1_in)
w1 = fibonacci_witness_fn(z1_in)
x1 = GF(np.concatenate([z1_in, z1_out]))

z2_in = GF(np.array([3, 5]))
z2_out = fibonacci_step_fn(z2_in)
w2 = fibonacci_witness_fn(z2_in)
x2 = GF(np.concatenate([z2_in, z2_out]))

# Both satisfy R1CS
print(f"Instance 1 satisfies R1CS: {shape.is_satisfied(x1, w1)}")
print(f"Instance 2 satisfies R1CS: {shape.is_satisfied(x2, w2)}")

In [ ]:
# Attempt naive folding: z' = z1 + r*z2
r = GF(7)  # pick a fixed r for reproducibility

full_z1 = GF(np.concatenate([GF(np.array([1])), x1, w1]))
full_z2 = GF(np.concatenate([GF(np.array([1])), x2, w2]))
z_combined = full_z1 + r * full_z2

print(f"z1 = {full_z1}")
print(f"z2 = {full_z2}")
print(f"z' = z1 + {int(r)}*z2 = {z_combined}")

# Check: does z' satisfy Az'∘Bz' = Cz' ?
Az = shape.A @ z_combined
Bz = shape.B @ z_combined
Cz = shape.C @ z_combined
lhs = Az * Bz
rhs = Cz

print(f"\nAz' ∘ Bz' = {lhs}")
print(f"Cz'       = {rhs}")
print(f"\nSatisfied? {np.all(lhs == rhs)}")
print("\n❌ It fails! Let's see why...")

## Why It Fails: The Cross-Term

Expanding $A(\mathbf{z}_1 + r\mathbf{z}_2) \circ B(\mathbf{z}_1 + r\mathbf{z}_2)$:

$$= A\mathbf{z}_1 \circ B\mathbf{z}_1 + r(A\mathbf{z}_1 \circ B\mathbf{z}_2 + A\mathbf{z}_2 \circ B\mathbf{z}_1) + r^2 \cdot A\mathbf{z}_2 \circ B\mathbf{z}_2$$

We need this to equal $C(\mathbf{z}_1 + r\mathbf{z}_2) = C\mathbf{z}_1 + r \cdot C\mathbf{z}_2$.

The **cross-term** $r(A\mathbf{z}_1 \circ B\mathbf{z}_2 + A\mathbf{z}_2 \circ B\mathbf{z}_1)$ has no corresponding term on the right side.

**Physics analogy:** This is exactly like **perturbation theory**. When you expand $(H_0 + \lambda V)^2$, you get the interference term $2\lambda H_0 V$ that you can't ignore.

In [ ]:
# Let's compute each term explicitly
Az1 = shape.A @ full_z1
Bz1 = shape.B @ full_z1
Az2 = shape.A @ full_z2
Bz2 = shape.B @ full_z2
Cz1 = shape.C @ full_z1
Cz2 = shape.C @ full_z2

# The three terms from expanding LHS
term_0 = Az1 * Bz1                      # r^0 term
term_1 = Az1 * Bz2 + Az2 * Bz1          # r^1 term (THE CROSS-TERM)
term_2 = Az2 * Bz2                      # r^2 term

print("LHS expansion:")
print(f"  r^0 term: Az1∘Bz1     = {term_0}")
print(f"  r^1 term: CROSS-TERM  = {term_1}  ← the problem!")
print(f"  r^2 term: Az2∘Bz2     = {term_2}")

print(f"\nRHS:")
print(f"  r^0 term: Cz1  = {Cz1}")
print(f"  r^1 term: Cz2  = {Cz2}")

# The error is the cross-term minus the expected r^1 contribution from C
error = term_1 - Cz2
print(f"\nError (cross-term - Cz2) = {error}")
print("This error is what Relaxed R1CS absorbs into E.")

## Relaxed R1CS: The Fix

Nova introduces **Relaxed R1CS**:

$$A\mathbf{z} \circ B\mathbf{z} = u \cdot C\mathbf{z} + \mathbf{E}$$

Two new components:
- **$u$** (scalar): "relaxation factor" — for real instances $u=1$, but after folding $u \neq 1$
- **$\mathbf{E}$** (vector): "error term" — absorbs the cross-terms from folding

For a genuine R1CS instance: $u=1$, $\mathbf{E}=\mathbf{0}$ → reduces to standard R1CS.

**Crucial detail:** In Relaxed R1CS, $\mathbf{z} = (u, \mathbf{x}, \mathbf{W})$ — the first element is $u$, not $1$. This ensures that when we fold $\mathbf{z}' = \mathbf{z}_1 + r\mathbf{z}_2$, the first element correctly becomes $u_1 + r \cdot u_2 = u'$.

In [ ]:
from nano_nova.commitment import commit
from nano_nova.r1cs import r1cs_to_relaxed, trivial_relaxed, RelaxedR1CSInstance, RelaxedR1CSWitness

# Lift our two R1CS instances to Relaxed R1CS
inst1, wit1 = r1cs_to_relaxed(shape, x1, w1, commit)
inst2, wit2 = r1cs_to_relaxed(shape, x2, w2, commit)

print("Instance 1 (relaxed):")
print(f"  u = {int(inst1.u)}, E = {wit1.E}, W = {wit1.W}")
print(f"  Satisfies relaxed R1CS: {shape.is_relaxed_satisfied(inst1, wit1)}")

print(f"\nInstance 2 (relaxed):")
print(f"  u = {int(inst2.u)}, E = {wit2.E}, W = {wit2.W}")
print(f"  Satisfies relaxed R1CS: {shape.is_relaxed_satisfied(inst2, wit2)}")

In [ ]:
# The trivial instance: u=0, E=0, W=0
# This is the starting point for IVC
triv_inst, triv_wit = trivial_relaxed(shape, commit)
print("Trivial instance:")
print(f"  u = {int(triv_inst.u)}, E = {triv_wit.E}, W = {triv_wit.W}")
print(f"  Satisfies relaxed R1CS: {shape.is_relaxed_satisfied(triv_inst, triv_wit)}")
print("\n  Why? With u=0 and E=0: LHS = Az∘Bz = 0 (since z starts with u=0),")
print("  and RHS = 0·Cz + 0 = 0. Both sides are zero. ✓")

## Key Takeaway

Relaxed R1CS ($A\mathbf{z} \circ B\mathbf{z} = u \cdot C\mathbf{z} + \mathbf{E}$) is the container that makes folding work:

1. **$u$ tracks scaling** — after folding, $u' = u_1 + r \cdot u_2 \neq 1$
2. **$\mathbf{E}$ absorbs cross-terms** — the "perturbative corrections" from combining quadratic constraints
3. **$\mathbf{z} = (u, \mathbf{x}, \mathbf{W})$** — the first element is $u$, not a fixed constant

Next: [04_folding_step.ipynb](04_folding_step.ipynb) — the actual folding algorithm, step by step.